# Test Dataset

In [1]:
!pip install mdformat
import requests
from bs4 import BeautifulSoup
from IPython.display import HTML, Markdown
from markdownify import markdownify
import os
import copy
import tiktoken
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser, SentenceWindowNodeParser
import pandas as pd
from transformers import AutoTokenizer
from llama_index.core import Document, set_global_tokenizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import MetadataMode
import numpy as np
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

    yapf (>='0.28') ; python_version < "3.6"
         ~^


## Preprocessing

### Wiki Page

In [2]:
def clean_wiki(soup):
    # Remove citation references like [1], [2]
    for sup in soup.find_all('sup', class_='reference'):
        sup.decompose()
    
    # Remove 'Edit' links
    for span in soup.find_all('span', class_='mw-editsection'):
        span.decompose()
    
    # Remove navigation boxes and tables at the bottom of the page
    for nav in soup.find_all('div', class_='navbox'):
        nav.decompose()
    
    # Remove <figure> tags (Wikipedia uses these for images + captions)
    for figure in soup.find_all('figure'):
        figure.decompose()
    
    # Remove Wikipedia-specific thumbnail div containers
    for thumb in soup.find_all('div', class_='thumb'):
        thumb.decompose()
    
    # Remove any remaining standalone <img> tags (e.g., small icons)
    for img in soup.find_all('img'):
        img.decompose()

    # Remove link
    for a in soup.find_all('a'):
        a.unwrap()

    # Remove bad sections
    sections_to_remove = ['See_also', 'References', 'External_links']
    for section_id in sections_to_remove:
        heading = soup.find(id=section_id)
        if heading:
            parent_section = heading.find_parent('section')
            if parent_section:
                parent_section.decompose()

    # Remove duplicated header
    if soup.head:
        soup.head.decompose()
    elif soup.title:
        soup.title.decompose()

In [3]:
url = "https://en.wikipedia.org/api/rest_v1/page/html/City_University_of_Hong_Kong"
headers = {
    "User-Agent": "MyApp/1.0 (myemail@example.com)"
}

# 1. Fetch the HTML
response = requests.get(url, headers=headers)
response.raise_for_status()

# 2. Parse the HTML
soup = BeautifulSoup(response.text, 'html.parser')

# 3. Extract the title BEFORE cleaning the soup (since we delete the <head> tag)
page_title = soup.title.text if soup.title else "Untitled"

# 4. Clean the HTML
clean_wiki(soup)

# 5. Convert the cleaned HTML to Markdown
markdown_text = markdownify(str(soup), heading_style="ATX")

# 6. Prepend the extracted title as an H1 heading
markdown_text = f"# {page_title}\n\n" + markdown_text

# 7. Save the result to a Markdown file
output_filename = "City_University_of_Hong_Kong.md"
with open(output_filename, "w", encoding="utf-8") as file:
    file.write(markdown_text)

print(f"Successfully saved to {output_filename}")

Successfully saved to City_University_of_Hong_Kong.md


### CS Curriculum

Headers are manually applied to the processed markdown file.

In [4]:
class CleanCurriculum():
    def __init__(self):
        self.cat_term = 'Catalogue Term : Semester A 2025/26'
    
    def __call__(self, soup):
        soup = self.keep_main_content(soup)

        table = soup.find('table')
        self.flatten_table(table)

        for table in soup.select('#reqsX'):
            self.flatten_table(table)

        self.remove_link(soup)
        self.remove_span_with_text(soup, "Catalogue Term : Semester A 2025/26")
        self.remove_hr(soup)
        
        return soup

    def keep_main_content(self, soup):
        content = soup.find("div", id="cityu-content")
        new_soup = BeautifulSoup(
            "<!DOCTYPE html><html><head></head><body></body></html>",
            "html.parser"
        )
        for child in list(content.children):
            new_soup.body.append(child.extract())

        return new_soup

    def remove_link(self, soup):
        for a in soup.find_all('a'):
            a.unwrap()

    def remove_hr(self, soup):
        for hr in soup.find_all('hr'):
            hr.decompose()

    def flatten_table(self, target_table):
        # 2. Create the new parent <div>
        container_div = soup.new_tag('div')
    
        # 3. Navigate the table structure using recursive=False to avoid nested tables
        table_sections = target_table.find_all(['tbody', 'thead', 'tfoot', 'tr'], recursive=False)
    
        for section in table_sections:
            if section.name == 'tr':
                rows = [section]
            else:
                rows = section.find_all('tr', recursive=False)
                
            for row in rows:
                for td in row.find_all('td', recursive=False):
                    inner_div = soup.new_tag('div')
                    
                    # Move contents into the new div
                    inner_div.extend(td.contents) 
                    container_div.append(inner_div)
    
        # 4. Replace the original table with the new container_div
        target_table.replace_with(container_div)

    def remove_span_with_text(self, soup, text):
        for span in soup.find_all('span'):
            span_text = span.get_text(strip=True)
            
            if text in span_text:
                span.decompose()

    def standardize_header(self, soup, _id, tag):
        target_divs = soup.select(f'strong > div#{_id}')

        for div in target_divs:
            h1_tag = soup.new_tag(tag)
            h1_tag.string = div.get_text(strip=True)
            div.parent.replace_with(h1_tag)

In [5]:
curriculum = None

with open('./CS_Curriculum_Detail_Raw.html', 'r', encoding='utf-8') as f:
    curriculum = f.read()

clean_curl = CleanCurriculum()
soup = BeautifulSoup(curriculum, 'html.parser')
clean_soup = clean_curl(soup)

with open("CS_Curriculum_Clean.html", "w", encoding="utf-8") as file:
    file.write(clean_soup.prettify())

with open("CS_Curriculum_Clean.md", "w", encoding="utf-8") as file:
    md = markdownify(str(clean_soup), heading_style="ATX", keep=['table'])
    file.write(md)

### Magazines

In [6]:
def merge_contents(contents):
    new_soup = BeautifulSoup(
            "<!DOCTYPE html><html><head></head><body></body></html>",
            "html.parser"
        )
    for child in list([child for content in contents for child in content.children]):
        new_soup.body.append(child.extract())

    return new_soup

In [7]:
def get_mag_contents(paths):
    mag_contents = []
    
    for fp in paths:
        with open(fp, 'r', encoding='utf-8') as f:
            mag = f.read()
            soup = BeautifulSoup(mag, 'html.parser')
            content = soup.select_one('.feature.pageCtn')
            mag_contents.append(content)

    return mag_contents

In [8]:
class CleanMagazine():    
    def __call__(self, soup):
        self.remove_img(soup)
        self.remove_link(soup)
        
        return soup

    def remove_img(self, soup):
        for img in soup.select('.pageImgCtn'):
            img.decompose()

    def remove_link(self, soup):
        for a in soup.find_all('a'):
            a.unwrap()

In [9]:
clean_mag = CleanMagazine()

mag_contents = get_mag_contents([f'City_Magazine_{i + 1}.html' for i in range(5)])
soup = merge_contents(mag_contents)
clean_soup = clean_mag(soup)

with open("City_Magazine_Clean.html", "w", encoding="utf-8") as file:
    file.write(soup.prettify())

with open("City_Magazine_Clean.md", "w", encoding="utf-8") as file:
    md = markdownify(str(clean_soup), heading_style="ATX", keep=['table'])
    file.write(md)

## Statistics

### Tokenizer

In [34]:
model_name = "BAAI/bge-small-en-v1.5"
embed_model = HuggingFaceEmbedding(model_name=model_name)
bge_tokenizer = AutoTokenizer.from_pretrained(model_name)
set_global_tokenizer(
    lambda text: bge_tokenizer.encode(text, add_special_tokens=False)
)

### Get Stats

In [35]:
def get_stats_chunk_sizes(nodes, use_window=False):
    chunk_sizes = []
    
    for node in nodes:
        if use_window and "window" in node.metadata:
            content = node.metadata["window"]
        else:
            content = node.get_content(metadata_mode=MetadataMode.NONE)
        tokens = bge_tokenizer.encode(content, add_special_tokens=False)
        chunk_sizes.append(len(tokens))

    total = sum(chunk_sizes)
    min_ = min(chunk_sizes)
    max_ = max(chunk_sizes)
    mean = np.mean(chunk_sizes)

    return total, min_, max_, mean

def get_stats(doc_wiki, doc_cur, doc_mag, splitter, use_window=False):
    nodes_wiki = splitter.get_nodes_from_documents([doc_wiki])
    nodes_cur = splitter.get_nodes_from_documents([doc_cur])
    nodes_mag = splitter.get_nodes_from_documents([doc_mag])

    chunk_sizes_wiki = get_stats_chunk_sizes(nodes_wiki, use_window)
    chunk_sizes_cur = get_stats_chunk_sizes(nodes_cur, use_window)
    chunk_sizes_mag = get_stats_chunk_sizes(nodes_mag, use_window)
    chunk_sizes = np.array([
        chunk_sizes_wiki,
        chunk_sizes_cur,
        chunk_sizes_mag
    ])

    stats = pd.DataFrame({
        "dataset": ["Wiki", "Curriculum", "Magazine"],
        "num_chunks": [len(nodes_wiki), len(nodes_cur), len(nodes_mag)],
        "total_chunk_size": chunk_sizes[:, 0],
        "mean_chunk_size": chunk_sizes[:, 3],
        "min_chunk_size": chunk_sizes[:, 1],
        "max_chunk_size": chunk_sizes[:, 2],
    })

    return stats

In [40]:
md_wiki = None
md_cur = None
md_mag = None

with open('./City_University_of_Hong_Kong.md', 'r', encoding='utf-8') as f:
    md_wiki = f.read()

with open('./CS_Curriculum_Clean.md', 'r', encoding='utf-8') as f:
    md_cur = f.read()

with open('./City_Magazine_Clean.md', 'r', encoding='utf-8') as f:
    md_mag = f.read()

doc_wiki = Document(text=md_wiki)
doc_cur = Document(text=md_cur)
doc_mag = Document(text=md_mag)

strategy_labels = {
    "SentenceSplitter(chunk_size=128, chunk_overlap=10)": "SS-128",
    "SentenceSplitter(chunk_size=256, chunk_overlap=10)": "SS-256",
    "SentenceSplitter(chunk_size=512, chunk_overlap=10)": "SS-512",
    "SemanticSplitterNodeParser": "Semantic",
    "SentenceWindowNodeParser": "SentWindow"
}

ss_128 = SentenceSplitter(chunk_size=128, chunk_overlap=10)
ss_128_stats = get_stats(doc_wiki, doc_cur, doc_mag, ss_128)
ss_128_stats['strategy'] = strategy_labels['SentenceSplitter(chunk_size=128, chunk_overlap=10)']

ss_256 = SentenceSplitter(chunk_size=256, chunk_overlap=10)
ss_256_stats = get_stats(doc_wiki, doc_cur, doc_mag, ss_256)
ss_256_stats['strategy'] = strategy_labels['SentenceSplitter(chunk_size=256, chunk_overlap=10)']

ss_512 = SentenceSplitter(chunk_size=512, chunk_overlap=10)
ss_512_stats = get_stats(doc_wiki, doc_cur, doc_mag, ss_512)
ss_512_stats['strategy'] = strategy_labels['SentenceSplitter(chunk_size=512, chunk_overlap=10)']

ssn = SemanticSplitterNodeParser(embed_model=embed_model)
ssn_stats = get_stats(doc_wiki, doc_cur, doc_mag, ssn)
ssn_stats['strategy'] = strategy_labels['SemanticSplitterNodeParser']

swn = SentenceWindowNodeParser()
swn_stats = get_stats(doc_wiki, doc_cur, doc_mag, swn, use_window=True)
swn_stats['strategy'] = strategy_labels['SentenceWindowNodeParser']

stats = pd.concat([ss_128_stats, ss_256_stats, ss_512_stats, ssn_stats, swn_stats])
stats = stats.reindex(columns=['strategy', 'dataset', 'num_chunks', 'total_chunk_size', 'mean_chunk_size', 'min_chunk_size', 'max_chunk_size']).reset_index(drop=True)
stats.loc[stats['strategy'].duplicated(), 'strategy'] = ''
print(stats.to_markdown(index=False, floatfmt=("", "", "", ".0f", ".2f", ".0f", ".0f")))

| strategy   | dataset    |   num_chunks |   total_chunk_size |   mean_chunk_size |   min_chunk_size |   max_chunk_size |
|:-----------|:-----------|-------------:|-------------------:|------------------:|-----------------:|-----------------:|
| SS-128     | Wiki       |           52 |               5761 |            110.79 |               59 |              128 |
|            | Curriculum |           19 |               2260 |            118.95 |               44 |              128 |
|            | Magazine   |           43 |               4676 |            108.74 |               55 |              128 |
| SS-256     | Wiki       |           24 |               5630 |            234.58 |              120 |              256 |
|            | Curriculum |           10 |               2178 |            217.80 |              133 |              256 |
|            | Magazine   |           20 |               4685 |            234.25 |              191 |              255 |
| SS-512     | Wiki     

In [41]:
stats[['num_chunks', 'total_chunk_size', 'mean_chunk_size', 'min_chunk_size', 'max_chunk_size']].mean()

num_chunks            32.733333
total_chunk_size    8755.200000
mean_chunk_size      448.098723
min_chunk_size       206.733333
max_chunk_size       744.933333
dtype: float64